In [1]:
import numpy as np
from bstrapping.weights.gaussian_weights import GaussianWeights

from bstrapping.bootstrap_procedures.weighted_bootstrap import WeightedBootstrap
from bstrapping.synthetic_time_series.moving_average import MovingAverage
from bstrapping.weights.auto_regressive_weights import AutoRegressiveWeights
from bstrapping.weights.moving_average import MovingAverageWeights
import pandas as pd

from typing import List

In [2]:
pd.set_option('display.max_columns', None)

# Testing different bootstrap procedures for a single sample

In [3]:
# specify variance, mean and number of the samples
mean = 4
number_sample_points = 2000

# generate samples from a normal distribution
time_series = MovingAverage(mean=mean, parameters=np.array([0.8]))
samples = time_series.generate_samples(number_samples=number_sample_points)
print(f"Estimated mean: {np.mean(samples)}")

Estimated mean: 3.977527876696807


In [4]:
# Benchmark
evaluation_bootstraps = []
for weights in [AutoRegressiveWeights(samples=samples),
                GaussianWeights(samples=samples),
                MovingAverageWeights(samples=samples)]:
    bootstrap = WeightedBootstrap(samples=samples, weights=weights, number_bootstrap_samples=100)
    # asymptotic variance and confidence interval
    evaluation_bootstraps.append([number_sample_points * bootstrap.bootstrapped_variance,
                                  bootstrap.bootstrapped_confidence_interval(alpha=0.05)])

evaluation_bootstraps = dict(zip(['AR', 'Multiplier', 'MA'], evaluation_bootstraps))

2000 samples with dimension 1 were obtained. 

Bootstrapping...


100%|██████████| 100/100 [00:00<00:00, 142.83it/s]


2000 samples with dimension 1 were obtained. 

Bootstrapping...


100%|██████████| 100/100 [00:00<00:00, 12594.37it/s]


2000 samples with dimension 1 were obtained. 

Bootstrapping...


100%|██████████| 100/100 [00:08<00:00, 11.48it/s]


In [5]:
print(
    f"True asymptotic variance: {time_series.asymptotic_variance}\nEstimated mean from samples: {np.mean(samples)}\nTrue mean: {mean}")

True asymptotic variance: 3.24
Estimated mean from samples: 3.977527876696807
True mean: 4


In [6]:
pd.DataFrame(evaluation_bootstraps)

,AR,Multiplier,MA
0,3.439848,1.43922,2.437786
1,"[3.8977782381269948, 4.059320680473192]","[3.928595515137408, 4.026906827749747]","[3.9102357495858353, 4.040266587043126]"


# Statistical evaluation of the bootstrap procedures for fixed sample size

In [7]:
runs = 100

In [8]:
%%capture
# Benchmark
bootstrapped_variances = [[], [], []]
for _ in range(runs):
    samples = time_series.generate_samples(number_samples=number_sample_points)
    for i, weights in enumerate([AutoRegressiveWeights(samples=samples),
                                 GaussianWeights(samples=samples),
                                 #MovingAverageWeights(samples=samples)
                                 ]):
        bootstrap = WeightedBootstrap(samples=samples, weights=weights, number_bootstrap_samples=100)
        # asymptotic variance and confidence interval
        bootstrapped_variances[i].append(number_sample_points * bootstrap.bootstrapped_variance,
                                         )

statistical_evaluation = [[np.mean(variance), np.std(variance)] for variance in bootstrapped_variances]
statistical_evaluation = dict(zip(['AR',
                                   'Multiplier',
                                   #'MA'
                                   ], statistical_evaluation))

In [9]:
print(f"Estimated mean: {np.mean(samples)}")

Estimated mean: 4.006302578210814


In [10]:
pd.DataFrame(statistical_evaluation, index=['mean', 'std'])

,AR,Multiplier
mean,3.179066,1.598395
std,0.558681,0.225614


# Statistical evaluation of the bootstrap procedures for increasing sample size

In [11]:
def evaluate_bootstraps(sample_size: int, runs: int = 100):
    bootstrapped_variances = [[], [], []]
    for _ in range(runs):
        samples = time_series.generate_samples(number_samples=sample_size)
        for i, weights in enumerate([AutoRegressiveWeights(samples=samples),
                                     GaussianWeights(samples=samples),
                                     #MovingAverageWeights(samples=samples)
                                     ]):
            bootstrap = WeightedBootstrap(samples=samples, weights=weights, number_bootstrap_samples=100)
            # asymptotic variance and confidence interval
            bootstrapped_variances[i].append(sample_size * bootstrap.bootstrapped_variance,
                                             )
    return bootstrapped_variances

In [12]:
def benchmark_bootstraps(sample_sizes: List[int], runs: int = 100):
    benchmark = []
    for sample_size in sample_sizes:
        benchmark.append(evaluate_bootstraps(sample_size=sample_size, runs=runs))

    return benchmark


In [29]:
%%capture
sample_sizes = [1000, 2000, 5000]
# of the form 0 index: sample sizes 1 index: bootstrap procedures 2 index: bootstrapped variances
plain_result = benchmark_bootstraps(sample_sizes=sample_sizes, runs=5)

In [30]:
list_name_weights = ['AR',
                     'Multiplier',
                     #'MA',
                     ]
statistical_evaluation = {
    (name, "mean"): [np.mean(plain_result[sample_size_index][name_index]) for
                                              sample_size_index in range(len(plain_result))]

    for name_index, name in enumerate(list_name_weights)
} | {
    (name, "std"): [np.std(plain_result[sample_size_index][name_index]) for
                                              sample_size_index in range(len(plain_result))]
    for name_index, name in enumerate(list_name_weights)
}

statistical_evaluation = pd.DataFrame(statistical_evaluation,index=sample_sizes)

In [31]:
statistical_evaluation

,AR,Multiplier,AR,Multiplier
,mean,mean,std,std
1000,3.781523,1.513305,1.072008,0.161147
2000,2.668187,1.592447,0.454865,0.105891
5000,3.353161,1.417451,0.354072,0.132308


# Statistical evaluation of the bootstrap procedures for increasing dependencies

In [ ]:
dependence_coefficients = [
    np.array([0.8]),
    np.array([0.6,0.8]),
]

In [ ]:
def evaluate_bootstrap_incresing_dependencies(dependence_coefficients: List[np.ndarray],sample_size: int = 1000, runs: int = 100):
    for dependence_coefficient in dependence_coefficients:
        time_series = MovingAverage(mean=mean, parameters=dependence_coefficient)
        evaluate_bootstraps(sample_size, runs)
        

In [16]:
#bootstrap.save_to_csv('discrete_bootstrap_iid_samples')